# Dialogue Summarization Project - Phase 2

## Classical NLP Baseline: TF-IDF Extractive Summarization

In Phase 1, we explored and preprocessed the SAMSum dataset.

In Phase 2, we build a simple baseline model using classical NLP.

This baseline uses:

- TF-IDF vectorization
- Sentence scoring
- Extractive summary generation
- ROUGE evaluation

A baseline model is important because it gives us a simple result to compare with more advanced models later.

## 1. Install Required Libraries

If you are using Google Colab, run the installation cell below once.

If you already installed the project requirements, you can skip this cell.

In [ ]:
# For Google Colab, uncomment and run this line if needed.
# !pip install datasets pandas numpy nltk scikit-learn rouge-score

## 2. Import Libraries

We import libraries for dataset loading, data handling, sentence tokenization, TF-IDF, and ROUGE evaluation.

In [ ]:
# Standard Python libraries
import os
import sys

# Data handling
import pandas as pd

# Dataset loading
from datasets import load_dataset

# Add src folder to Python path so we can import our own code.
sys.path.append("src")

# Import functions from our baseline model file.
from train_baseline import (
    download_nltk_data,
    split_dialogue_into_sentences,
    score_sentences_with_tfidf,
    generate_extractive_summary,
    calculate_rouge_scores,
    create_predictions_dataframe,
    create_rouge_summary,
    create_comparison_table,
)


# Import preprocessing helpers for Week 2 roadmap checks.
from preprocess import clean_text, remove_punctuation, tokenize_text

# Create results folder if it does not already exist.
os.makedirs("results", exist_ok=True)

# Download NLTK sentence tokenizer data.
download_nltk_data()

print("Libraries imported successfully!")

## 3. Load the SAMSum Dataset

We use the same dataset from Phase 1.

For this baseline, we evaluate on the `test` split because test data is used for final model evaluation.

In [ ]:
# Load the SAMSum dataset from Hugging Face.
dataset = load_dataset("knkarthick/samsum")

# Convert the test split to a pandas DataFrame.
test_df = dataset["test"].to_pandas()

# Show the shape and first few rows.
print("Test samples:", len(test_df))
test_df.head()

## 4. Text Cleaning and Tokenization Checks

This section demonstrates the Week 2 preprocessing requirements before building the TF-IDF baseline.


In [ ]:
# Apply clean_text() to real dialogue examples and show before/after cleaning.
cleaning_examples = test_df[["id", "dialogue"]].head(3).copy()
cleaning_examples["cleaned_dialogue"] = cleaning_examples["dialogue"].apply(clean_text)

# This custom example makes URL, HTML, emoticon, lowercase, punctuation, and spacing behavior visible.
custom_dirty_text = "<b>Hello THERE!!!</b> Visit https://example.com now :)   Thanks!!!"
custom_clean_text = clean_text(custom_dirty_text)
custom_no_punctuation = remove_punctuation(custom_clean_text)

print("Custom text cleaning example:")
print("Before:", custom_dirty_text)
print("After clean_text():", custom_clean_text)
print("After punctuation removal:", custom_no_punctuation)

cleaning_examples[["id", "dialogue", "cleaned_dialogue"]]


## 5. NLTK Word Tokenization Example


In [ ]:
# tokenize_text() uses NLTK word_tokenize after cleaning and punctuation handling.
tokens = tokenize_text(custom_dirty_text)

print("NLTK word_tokenize output:")
print(tokens)


## 6. What Is Extractive Summarization?

Extractive summarization selects important sentences from the original text.

For example, if a dialogue has 10 sentences, an extractive summarizer may choose the 3 most important sentences and join them together.

This is simpler than abstractive summarization, where a model writes new sentences.

## 7. Sentence Splitting Example

Before scoring sentences, we first split a dialogue into sentence-like pieces.

SAMSum dialogues often have speaker names and line breaks, so our function handles each line separately.

In [ ]:
# Select one dialogue example.
example_dialogue = test_df.iloc[0]["dialogue"]
example_reference = test_df.iloc[0]["summary"]

# Split the dialogue into sentences.
example_sentences = split_dialogue_into_sentences(example_dialogue)

print("Original Dialogue:\n")
print(example_dialogue)

print("\nReference Summary:\n")
print(example_reference)

print("\nExtracted Sentences:\n")
for index, sentence in enumerate(example_sentences, start=1):
    print(index, sentence)

## 8. What Is TF-IDF?

TF-IDF stands for **Term Frequency - Inverse Document Frequency**.

In simple words:

- A word gets a higher score if it is important in a sentence.
- Very common words like `the`, `is`, and `and` are usually less useful.
- We can use TF-IDF scores to decide which sentences contain important words.

## 9. Score Sentences with TF-IDF

We treat each sentence as a small document.

Then we calculate a TF-IDF score for every sentence and rank them.

In [ ]:
# Score every sentence in the example dialogue.
scored_sentences = score_sentences_with_tfidf(example_sentences)

# Convert scores to a DataFrame so they are easy to view.
scores_df = pd.DataFrame(
    scored_sentences,
    columns=["sentence_index", "sentence", "tfidf_score"]
)

# Sort from highest score to lowest score.
scores_df.sort_values("tfidf_score", ascending=False)

## 10. Generate One Extractive Summary

Now we use the highest-scoring sentences to generate a simple summary.

The selected sentences are placed back in their original order so the summary is easier to read.

In [ ]:
# Generate a summary using the top 3 sentences.
generated_summary = generate_extractive_summary(
    dialogue=example_dialogue,
    num_sentences=3,
)

print("Generated Summary:\n")
print(generated_summary)

print("\nReference Summary:\n")
print(example_reference)

## 11. Evaluate One Prediction with ROUGE

ROUGE is a common metric for summarization.

We calculate:

- **ROUGE-1**: overlap of single words
- **ROUGE-2**: overlap of two-word phrases
- **ROUGE-L**: longest matching word sequence

Higher scores are better.

In [ ]:
# Calculate ROUGE scores for one example.
single_rouge_scores = calculate_rouge_scores(
    reference_summary=example_reference,
    generated_summary=generated_summary,
)

single_rouge_scores

## 12. Run Baseline on Multiple Test Samples

To keep this beginner-friendly and fast, we evaluate the first 100 test samples.

You can increase `MAX_SAMPLES` later if you want to evaluate more examples.

In [ ]:
# Number of test examples to evaluate.
MAX_SAMPLES = 100

# Number of sentences to extract for each summary.
NUM_SENTENCES = 3

# Generate summaries and calculate ROUGE scores.
predictions_df = create_predictions_dataframe(
    dataframe=test_df,
    num_sentences=NUM_SENTENCES,
    max_samples=MAX_SAMPLES,
)

predictions_df.head()

## 13. Evaluation Metrics

Now we calculate the average ROUGE scores across all evaluated samples.

In [ ]:
# Average ROUGE scores across all predictions.
rouge_scores_df = create_rouge_summary(predictions_df)
rouge_scores_df

## 14. Comparison Table

This table is useful for reports.

For now, it contains only one baseline model. In future phases, we can add transformer models and compare them with this baseline.

In [ ]:
# Create a comparison table for report use.
comparison_table_df = create_comparison_table(
    rouge_df=rouge_scores_df,
    max_samples=len(predictions_df),
)

comparison_table_df

## 15. View Sample Predictions

Let us compare generated summaries with human-written summaries.

This helps us understand what the baseline is doing well and where it is weak.

In [ ]:
# Display a few sample predictions.
sample_view = predictions_df[
    ["id", "reference_summary", "generated_summary", "rouge1", "rouge2", "rougeL"]
].head(10)

sample_view

## 16. Save Result Files

We save three CSV files in the `results/` folder:

- `rouge_scores.csv`
- `sample_predictions.csv`
- `comparison_table.csv`

In [ ]:
# Save generated predictions.
predictions_df.to_csv("results/sample_predictions.csv", index=False)

# Save average ROUGE scores.
rouge_scores_df.to_csv("results/rouge_scores.csv", index=False)

# Save comparison table.
comparison_table_df.to_csv("results/comparison_table.csv", index=False)

print("Result files saved successfully in the results/ folder!")

## 17. Phase 2 Summary

In this notebook, we completed Phase 2:

- Built a TF-IDF based extractive summarizer
- Scored dialogue sentences using TF-IDF
- Generated summaries from top-scoring sentences
- Evaluated predictions with ROUGE-1, ROUGE-2, and ROUGE-L
- Saved predictions, metrics, and comparison table in the `results/` folder

This baseline is simple, but it gives us a useful starting point for comparing future deep learning models.